# 🧠 Advanced Graph Reasoning Demo

This notebook demonstrates **5 advanced graph reasoning query types** that require sophisticated multi-hop traversal and inference. These queries highlight the strength of knowledge graph-based retrieval.

## Query Types Covered:
1. **Multi-Hop Chain Reasoning** - Follow chains of relationships with constraints
2. **Subgraph Pattern Matching** - Find images matching complex patterns
3. **Scene Comparison** - Compare structural properties of two scenes
4. **Counterfactual Reasoning** - "What if" analysis
5. **Centrality-Based Queries** - Find influential nodes

In [ ]:
# Setup: Import the Advanced Reasoning Engine
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'reasoning' else Path.cwd()
sys.path.insert(0, str(project_root))

from reasoning.advanced_reasoning_engine import AdvancedReasoningEngine

print("✅ Imports successful!")

In [ ]:
# Initialize the Advanced Reasoning Engine
# Using 10k scale for demonstration (adjust based on available graphs)

engine = AdvancedReasoningEngine(scale='10k')
print(f"\n✅ Engine initialized with {engine.graph.number_of_nodes():,} nodes and {engine.graph.number_of_edges():,} edges")

---
## 1️⃣ Multi-Hop Chain Reasoning

**Purpose**: Answer questions that require traversing multiple relationship hops with intermediate constraints.

**Example**: Find images where `person → wearing → shirt → to the left of → tree`

This requires:
1. Finding all `person` instances
2. Traversing `wearing` relationships to find connected objects
3. Filtering to keep only `shirt` objects
4. Traversing `to the left of` relationships
5. Filtering to keep only `tree` objects

In [ ]:
# Example 1: Simple 2-hop chain
# Find: person → wearing → shirt → near → car

result = engine.chain_reasoning(
    start_concept="person",
    chain=[
        {"relation": "wearing", "concept": "shirt"},
        {"relation": "to the left of", "concept": "tree"}
    ],
    limit=5
)

result.print_result(verbose=True)

In [ ]:
# Example 2: Chain with attribute constraint
# Find: person → wearing → RED shirt → to the left of → building

result = engine.chain_reasoning(
    start_concept="person",
    chain=[
        {"relation": "wearing", "concept": "shirt", "attribute": "white"},
        {"relation": "to the left of", "concept": "building"}
    ],
    limit=5
)

result.print_result(verbose=True)

In [ ]:
# Example 3: 3-hop chain reasoning
# Find: man → holding → food → on → table

result = engine.chain_reasoning(
    start_concept="man",
    chain=[
        {"relation": "holding"},
        {"relation": "on", "concept": "table"}
    ],
    limit=5
)

result.print_result(verbose=True)

---
## 2️⃣ Subgraph Pattern Matching

**Purpose**: Find all images containing a specific relational pattern (mini-graph).

**Example Pattern**:
```
    person ──wearing──> shirt
      │
      └──wearing──> hat
```

This finds images where a person is wearing both a shirt AND a hat.

In [ ]:
# Example 1: Find person wearing both shirt and hat

result = engine.pattern_matching(
    pattern_nodes=["person", "shirt", "hat"],
    pattern_edges=[
        ("person", "wearing", "shirt"),
        ("person", "wearing", "hat")
    ],
    limit=10
)

result.print_result(verbose=True)

In [ ]:
# Example 2: Find car next to tree next to building (chain pattern)

result = engine.pattern_matching(
    pattern_nodes=["car", "tree", "building"],
    pattern_edges=[
        ("car", "to the left of", "tree"),
        ("tree", "to the left of", "building")
    ],
    limit=10
)

result.print_result(verbose=True)

In [ ]:
# Example 3: Find man with woman nearby (social scene)

result = engine.pattern_matching(
    pattern_nodes=["man", "woman"],
    pattern_edges=[
        ("man", "to the left of", "woman")
    ],
    limit=10
)

result.print_result(verbose=True)

---
## 3️⃣ Scene Comparison Reasoning

**Purpose**: Compare two images based on their graph structure.

**Metrics computed**:
- Common objects/concepts
- Unique objects to each image
- Jaccard similarity (concepts, attributes, relations)
- Overall structural similarity

In [ ]:
# Get some image IDs from the graph
sample_images = list(engine._image_to_objects.keys())[:10]
print("Sample images available:")
for i, img_id in enumerate(sample_images):
    obj_count = len(engine._image_to_objects[img_id])
    print(f"  {i+1}. Image {img_id}: {obj_count} objects")

In [ ]:
# Compare two images
if len(sample_images) >= 2:
    result = engine.scene_comparison(
        image_id_1=sample_images[0],
        image_id_2=sample_images[1]
    )
    
    result.print_result(verbose=True)
else:
    print("Need at least 2 images for comparison")

In [ ]:
# Compare different pair
if len(sample_images) >= 4:
    result = engine.scene_comparison(
        image_id_1=sample_images[2],
        image_id_2=sample_images[3]
    )
    
    result.print_result(verbose=True)

---
## 4️⃣ Counterfactual Reasoning

**Purpose**: Explore hypothetical scenarios by modifying the graph structure.

**Questions answered**:
- "What if we removed all `person` objects from this image?"
- "Which relationships would be affected?"
- "What scenes would be similar to this hypothetical version?"

In [ ]:
# Example 1: What if we removed 'person' from an image?

if sample_images:
    result = engine.counterfactual_reasoning(
        image_id=sample_images[0],
        remove_concept="person"
    )
    
    result.print_result(verbose=True)

In [ ]:
# Example 2: What if we added 'dog' to an image?
# This analyzes what relationships a dog typically has
# and predicts how it might interact with existing objects

if sample_images:
    result = engine.counterfactual_reasoning(
        image_id=sample_images[0],
        add_concept="dog"
    )
    
    result.print_result(verbose=True)

In [ ]:
# Example 3: Combined - remove one thing, add another

if sample_images:
    result = engine.counterfactual_reasoning(
        image_id=sample_images[0],
        remove_concept="tree",
        add_concept="car"
    )
    
    result.print_result(verbose=True)

---
## 5️⃣ Centrality-Based Queries

**Purpose**: Find important/influential nodes using graph centrality measures.

**Centrality Types**:
- `degree`: Count of connections (most connected nodes)
- `pagerank`: Importance based on incoming links
- `hub_images`: Images that connect many different concepts

In [ ]:
# Example 1: Most connected concepts (degree centrality)

result = engine.centrality_query(
    centrality_type="degree",
    node_filter="concept",
    top_k=15
)

result.print_result(verbose=True)

In [ ]:
# Example 2: Most connected attributes

result = engine.centrality_query(
    centrality_type="degree",
    node_filter="attribute",
    top_k=15
)

result.print_result(verbose=True)

In [ ]:
# Example 3: Hub images - images that connect many concepts

result = engine.centrality_query(
    centrality_type="hub_images",
    top_k=10
)

result.print_result(verbose=True)

In [ ]:
# Example 4: PageRank centrality for concepts

result = engine.centrality_query(
    centrality_type="pagerank",
    node_filter="concept",
    top_k=15
)

result.print_result(verbose=True)

---
## 📊 Summary: Reasoning Complexity

| Query Type | Graph Operations | Complexity |
|------------|------------------|------------|
| Chain Reasoning | Multi-hop traversal + filtering | O(n^k) |
| Pattern Matching | Subgraph isomorphism | O(n^p) |
| Scene Comparison | Jaccard similarity | O(e₁ + e₂) |
| Counterfactual | Virtual graph modification | O(base query) |
| Centrality | Graph metrics computation | O(V + E) |

These queries demonstrate the power of knowledge graph reasoning compared to simple keyword search!

In [ ]:
# Final summary
print("\n" + "="*60)
print("🎉 DEMO COMPLETE!")
print("="*60)
print(f"\nGraph Statistics:")
print(f"  • Total nodes: {engine.graph.number_of_nodes():,}")
print(f"  • Total edges: {engine.graph.number_of_edges():,}")
print(f"  • Concept nodes: {len(engine._concept_nodes):,}")
print(f"  • Attribute nodes: {len(engine._attribute_nodes):,}")
print(f"  • Instance nodes: {len(engine._instance_nodes):,}")
print(f"  • Images: {len(engine._image_to_objects):,}")
print(f"  • Relation types: {len(engine._relation_index):,}")
print("\nAdvanced queries available:")
print("  1. chain_reasoning() - Multi-hop traversal")
print("  2. pattern_matching() - Subgraph patterns")
print("  3. scene_comparison() - Compare images")
print("  4. counterfactual_reasoning() - What-if analysis")
print("  5. centrality_query() - Find important nodes")